In [1]:
%pip install yfinance

import yfinance as yf

ticker = yf.Ticker("cmg")

price = ticker.history(start="2009-02-14", end="2020-06-11")
print(price)

Note: you may need to restart the kernel to use updated packages.
                                Open       High        Low      Close  \
Date                                                                    
2009-02-17 00:00:00-05:00   1.097600   1.098800   1.057000   1.067200   
2009-02-18 00:00:00-05:00   1.069800   1.093000   1.048600   1.089000   
2009-02-19 00:00:00-05:00   1.106200   1.120000   1.068800   1.074800   
2009-02-20 00:00:00-05:00   1.054000   1.118000   1.048200   1.110000   
2009-02-23 00:00:00-05:00   1.121400   1.134200   1.086000   1.093400   
...                              ...        ...        ...        ...   
2020-06-04 00:00:00-04:00  20.886200  21.196199  20.700001  20.831200   
2020-06-05 00:00:00-04:00  21.000000  21.382000  20.901800  21.069201   
2020-06-08 00:00:00-04:00  21.056601  21.299999  20.664400  20.984600   
2020-06-09 00:00:00-04:00  20.738199  21.059799  20.700001  20.789400   
2020-06-10 00:00:00-04:00  20.852400  20.858801  20.469801

In [2]:
#Collect Features, we want them a day behind, we give yesturdays stats to guess todays price, so score follows current day
price["Change"] = price["Close"].shift(1) - price["Open"].shift(1)
price["Score"] = (price["Close"] > price["Open"]).astype(int)
price["%Change"] = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
price["YestScore"] = price["Score"].shift(1)
price["5DayMean"] = price["Close"].rolling(5).mean().shift(1)
price["5DayVoli"] = price["Close"].rolling(5).std().shift(1)
price["5DayPerf"] = price["Score"].rolling(5).sum().shift(1)

price.dropna(inplace=True)

print(price["5DayPerf"])

Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    3.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    2.0
                            ... 
2020-06-04 00:00:00-04:00    3.0
2020-06-05 00:00:00-04:00    3.0
2020-06-08 00:00:00-04:00    3.0
2020-06-09 00:00:00-04:00    2.0
2020-06-10 00:00:00-04:00    2.0
Name: 5DayPerf, Length: 2844, dtype: float64


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

priceTrain = price[price.index <= "2018-06-11"]
priceTest = price[price.index > "2018-06-11"]

results = []

for i in range(len(priceTest)):
    xTrain = priceTrain[features]
    yTrain = priceTrain["Score"]

    scaler = StandardScaler()
    xTrain = scaler.fit_transform(xTrain)

    model = LogisticRegression()
    model.fit(xTrain, yTrain)

    #Grab yesturdays information to predict todays, data already transformed so i is yesturday
    yest = priceTest.iloc[[i]]
    yestScal = scaler.transform(yest[features])
    pred = model.predict(yestScal)[0]
    prob = model.predict_proba(yestScal)[0][1]
    print("Analyzing Day: ", priceTest.index[i])

    results.append({"Date": priceTest.index[i],
                    "Score": priceTest["Score"].iloc[i],
                    "Prediction": pred,
                    "Probability": prob,
                    "Open": priceTest["Open"].iloc[i],
                    "Close": priceTest["Close"].iloc[i]})
    
    priceTrain = pd.concat([priceTrain, yest])

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [4]:
results = pd.DataFrame(results)

score = results["Score"]
pred = results["Prediction"]
op = results["Open"]
close = results["Close"]

accuracy = accuracy_score(score, pred)
precision = precision_score(score, pred, zero_division=0)
recall = recall_score(score, pred, zero_division=0)
f1 = f1_score(score, pred, zero_division=0)

print("Accuracy: ", accuracy, "\nPrecision: ", precision, "\nRecall: ", recall, "\nF1: ", f1)



Accuracy:  0.48111332007952284 
Precision:  0.46511627906976744 
Recall:  0.23809523809523808 
F1:  0.31496062992125984


In [5]:
#Find out how much you would have made/lost with this method
totalIn = 0
totalOut = 0
totalInvested = 0
totalStoodOut = 0
alltradein = 0
alltradeout = 0
perftradein = 0
perftradeout = 0
for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
    if i == 1:
        totalIn += 100
        totalInvested += 1
        temp = ((c - o)/o) * 100
        totalOut += 100 + temp
        print(temp, o, c)
    else:
        totalStoodOut += 1
    alltradein += 100
    alltradeout += (((c-o)/o) * 100) + 100
    if s == 1:
        perftradein += 100
        perftradeout += (((c-o)/o) * 100) + 100

print(totalIn)
print(totalOut)
print(totalInvested)
print(totalStoodOut)
print(alltradein)
print(alltradeout)
print(perftradein)
print(perftradeout)

-0.030014447008348842 9.328800201416016 9.326000213623047
-2.058209450916746 9.338199615478516 9.145999908447266
0.24383764407656097 9.186400413513184 9.208800315856934
1.0596609228827092 9.28600025177002 9.384400367736816
-1.1609041549144858 9.371999740600586 9.263199806213379
0.15985362046752383 9.38379955291748 9.398799896240234
1.3700510318176222 9.109199523925781 9.234000205993652
-1.4244652850916706 8.690999984741211 8.56719970703125
3.4674478248357095 8.559599876403809 8.856399536132812
0.26453147634130797 8.996999740600586 9.02079963684082
0.9807000703343758 9.054800033569336 9.143600463867188
-0.30309209039859364 9.17199993133545 9.144200325012207
-0.6810822011907939 9.10319995880127 9.041199684143066
0.2769901049603893 9.025799751281738 9.050800323486328
0.4039642113203241 9.0600004196167 9.096599578857422
1.2102888504475586 9.022600173950195 9.131799697875977
-1.634749517266942 9.138999938964844 8.98960018157959
1.260062000267707 8.92020034790039 9.032600402832031
-0.5564727